# Mini Data Quality Audit – Week 2 Individual Exercise

**Course:** Data Management – 2nd Semester  
**Date:** 2026-04-26  
**Dataset:** `week2_customer_transactions_messy.csv`

> **Environment:** Activate the virtual environment before launching Jupyter:  
> `.venv\Scripts\activate.bat`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
data_path = Path('week2_customer_transactions_messy.csv')
df = pd.read_csv(data_path)
print(f"Dataset loaded: {data_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df

## Task 1 – Dataset Description

### Business Context

This dataset contains **customer transaction records** for an e-commerce or payments
business operating across European and North American markets (DE, FR, NL, US).
Each row represents one financial transaction.

| Column | Type | Description |
|---|---|---|
| `transaction_id` | string | Unique identifier per transaction |
| `customer_id` | string | Reference to the customer account |
| `transaction_date` | string (date) | Date the transaction was initiated |
| `amount` | numeric | Transaction value in the stated currency |
| `currency` | string | ISO 4217 currency code |
| `payment_method` | string | Payment channel (card / bank_transfer / cash) |
| `status` | string | Lifecycle state: completed / pending / cancelled |
| `region` | string | ISO 3166-1 alpha-2 country code |
| `last_updated` | string (date) | Date of most recent record modification |

**Business use cases:**
- Revenue reporting and period-end reconciliation
- Customer segmentation and lifetime-value analytics
- Fraud / anomaly detection
- Payment-channel performance monitoring

The 11-row sample is small but representative: its quality issues mirror
real-world ETL pipeline failures across all five data-quality dimensions.

In [ ]:
# Quick profiling
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values per Column ===")
print(df.isnull().sum())
print("\n=== Unique Values per Column ===")
for col in df.columns:
    print(f"  {col}: {df[col].dropna().unique().tolist()}")

## Task 2 – Data Quality Issues by Dimension

In [ ]:
# ---------- helper ----------
def is_missing(series):
    return series.isna() | (series.astype(str).str.strip() == '')

# ---------- UNIQUENESS ----------
dup_mask = df.duplicated(subset=['transaction_id'], keep=False)

# ---------- VALIDITY ----------
amount_num  = pd.to_numeric(df['amount'], errors='coerce')
date_parsed = pd.to_datetime(
    df['transaction_date'].str.strip(), errors='coerce', format='mixed'
)
VALID_CURRENCIES = {'EUR', 'USD', 'GBP', 'CHF', 'JPY', 'CAD', 'AUD', 'NZD'}
invalid_currency = ~df['currency'].str.upper().str.strip().isin(VALID_CURRENCIES)
extreme_amount   = amount_num > 100_000

# ---------- CONSISTENCY ----------
non_iso_date = (
    df['transaction_date'].str.contains('/', na=False) |
    df['transaction_date'].str.match(r'^\d{2}-\d{2}-\d{4}', na=False)
)
region_not_upper  = df['region'].notna() & (df['region'] != df['region'].str.upper())
payment_not_lower = (
    df['payment_method'].notna() &
    (df['payment_method'] != df['payment_method'].str.lower())
)

# ---------- INTEGRITY (temporal) ----------
lu_parsed  = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
update_gap = (lu_parsed - date_parsed).dt.days
suspicious_lag = (update_gap > 60) & update_gap.notna()

# ---------- Issue table ----------
issue_rows = [
    ['Missing customer_id',                 'Completeness', int(is_missing(df['customer_id']).sum()),   'High'],
    ['Missing amount',                       'Completeness', int(is_missing(df['amount']).sum()),         'High'],
    ['Missing payment_method',               'Completeness', int(is_missing(df['payment_method']).sum()), 'Medium'],
    ['Missing last_updated',                 'Completeness', int(is_missing(df['last_updated']).sum()),   'Low'],
    ['Duplicate transaction_id',             'Uniqueness',   int(dup_mask.sum()),                         'High'],
    ['Negative amount',                      'Validity',     int((amount_num < 0).sum()),                 'High'],
    ['Invalid / impossible date',            'Validity',     int(date_parsed.isna().sum()),               'High'],
    ['Non-ISO-4217 currency (e.g. EURO)',    'Validity',     int(invalid_currency.sum()),                 'Medium'],
    ['Extreme outlier amount (> 100 000)',   'Validity',     int(extreme_amount.sum()),                   'Medium'],
    ['Inconsistent date format',             'Consistency',  int(non_iso_date.sum()),                     'Medium'],
    ['Region code not uppercase',            'Consistency',  int(region_not_upper.sum()),                 'Low'],
    ['Payment method not lowercase',         'Consistency',  int(payment_not_lower.sum()),                'Low'],
    ['Suspicious update lag (> 60 days)',    'Integrity',    int(suspicious_lag.sum()),                   'Medium'],
]
issue_table = pd.DataFrame(issue_rows, columns=['Issue', 'Dimension', 'Affected Rows', 'Severity'])
print(f'Total distinct issues: {len(issue_table)}')
issue_table

## Task 3 – KPI Calculations

In [ ]:
n_rows = len(df)
n_cols = len(df.columns)

# KPI 1 – Completeness Rate: fraction of non-null cells
total_missing     = df.isnull().sum().sum()
completeness_rate = 1 - (total_missing / (n_rows * n_cols))

# KPI 2 – Duplication Rate: duplicate rows on transaction_id
dup_rate = df.duplicated(subset=['transaction_id']).sum() / n_rows

# KPI 3 – Validity Error Rate: records with at least one validity issue
has_validity_issue  = (amount_num < 0) | date_parsed.isna() | invalid_currency | extreme_amount
validity_error_rate = has_validity_issue.mean()

# KPI 4 – Format Consistency Score: records with no format violations
format_issue      = non_iso_date | region_not_upper | payment_not_lower
format_consistency = 1 - format_issue.mean()

kpi_df = pd.DataFrame([
    ['Completeness Rate',        completeness_rate,   '>= 98%', completeness_rate >= 0.98],
    ['Duplication Rate',         dup_rate,            '0%',     dup_rate == 0.0],
    ['Validity Error Rate',      validity_error_rate, '0%',     validity_error_rate == 0.0],
    ['Format Consistency Score', format_consistency,  '>= 95%', format_consistency >= 0.95],
], columns=['KPI', 'Value', 'Target', 'Meets Target'])
kpi_df['Value (%)']    = kpi_df['Value'].apply(lambda x: f'{x*100:.1f}%')
kpi_df['Meets Target'] = kpi_df['Meets Target'].map({True: 'PASS', False: 'FAIL'})
kpi_df[['KPI', 'Value (%)', 'Target', 'Meets Target']]

### KPI Interpretation

| KPI | Value | Interpretation |
|---|---|---|
| **Completeness Rate** | ~96.0% | 4 of 99 cells are null — below the 98% target. Missing `customer_id` and `amount` are critical for revenue attribution and must be resolved first. |
| **Duplication Rate** | ~9.1% | One duplicate row (T0006 appears twice). Even a single duplicate in a transaction ledger directly inflates reported revenue, making this High-priority regardless of its low count. |
| **Validity Error Rate** | ~36.4% | Over one-third of records contain at least one critical validity defect (negative amount, impossible date, invalid currency, or extreme outlier). This is the strongest signal that the data pipeline has no input validation. |
| **Format Consistency Score** | ~81.8% | About one in five records use non-standard formats. Although less severe than validity errors, inconsistent formats cause silent misinterpretation in ETL and complicate downstream querying. |

## Task 4 – Validation Rules

In [ ]:
# Rule 1 – customer_id must be present
rule_customer_id_required = is_missing(df['customer_id'])

# Rule 2 – amount must be parseable and non-negative
rule_amount_non_negative  = (amount_num < 0) | amount_num.isna()

# Rule 3 – transaction_date must represent a valid calendar date
rule_date_valid           = date_parsed.isna()

# Rule 4 – currency must be a valid ISO 4217 code
rule_currency_iso4217     = invalid_currency

# Rule 5 – transaction_id must be unique across the dataset
rule_txn_id_unique        = df.duplicated(subset=['transaction_id'], keep=False)

rules = {
    'customer_id_required':  rule_customer_id_required,
    'amount_non_negative':   rule_amount_non_negative,
    'date_valid':            rule_date_valid,
    'currency_iso4217':      rule_currency_iso4217,
    'transaction_id_unique': rule_txn_id_unique,
}

val_summary = pd.DataFrame({
    'rule_name':      list(rules.keys()),
    'affected_rows':  [int(m.sum()) for m in rules.values()],
    'violation_rate': [f'{m.mean()*100:.1f}%' for m in rules.values()],
}).sort_values('affected_rows', ascending=False).reset_index(drop=True)
val_summary

In [ ]:
# Records that fail at least one rule
any_violation = pd.Series(False, index=df.index)
for mask in rules.values():
    any_violation = any_violation | mask

flagged = df[any_violation].copy()
flagged['violation_flags'] = [
    ', '.join(name for name, mask in rules.items() if mask[i])
    for i in flagged.index
]
print(f'Flagged records: {len(flagged)} / {len(df)}')
flagged

## Task 5 – Audit Summary

In [ ]:
audit_summary = pd.DataFrame([
    ['Missing customer_id',                        1, 'High',   'Retrieve from source system; block from revenue reports until resolved'],
    ['Duplicate transaction record (T0006)',        2, 'High',   'De-duplicate on transaction_id keeping row with latest last_updated; investigate ETL for double-processing'],
    ['Negative amount (T0003: -35.00 USD)',         1, 'High',   'Classify as refund or data-entry error; create refund_type field or separate refunds table'],
    ['Impossible date (T0007: 2026-02-30)',         1, 'High',   'Correct via source system; add calendar-validity check at ingestion'],
    ['Missing amount (T0009)',                      1, 'High',   'Retrieve from payment processor logs; exclude from revenue aggregations until resolved'],
    ['Invalid currency code (T0005: EURO)',         1, 'Medium', 'Map EURO -> EUR via ISO 4217 lookup; enforce validation at ETL boundary'],
    ['Inconsistent date formats (T0002, T0003)',    2, 'Medium', 'Normalise all dates to ISO 8601 (YYYY-MM-DD) in ingestion layer'],
    ['Missing payment_method (T0010)',              1, 'Medium', 'Impute from customer transaction history or mark as unknown'],
    ['Extreme amount outlier (T0008: 1 000 000)',   1, 'Medium', 'Verify with payment processor; route to review queue before including in reports'],
    ['Case inconsistency – region/payment (T0002)', 1, 'Low',   'Normalise: region uppercase, payment_method lowercase in ETL transform'],
    ['Suspicious update lag > 60 days (T0006)',     2, 'Low',   'Investigate reason for late modification; document in correction log'],
], columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])
audit_summary

## Task 6 – Recommended Cleaning Steps for Next Chapter

*These are **recommendations only**. Full cleaning pipelines will be implemented in Chapter 3.*

1. **Remove exact duplicates**  
   Sort by `last_updated` descending, then apply
   `df.drop_duplicates(subset=['transaction_id'], keep='first')`.
   This retains the most recently modified record when `transaction_id` is duplicated.

2. **Standardise date formats**  
   Parse with `pd.to_datetime(errors='coerce', format='mixed')`, then output via
   `.dt.strftime('%Y-%m-%d')`. Records that yield `NaT` (e.g., 2026-02-30) should be
   routed to a **quarantine table** — not silently deleted.

3. **Handle negative amounts**  
   Do not delete. Investigate whether they are refunds, chargebacks, or data-entry
   errors. If refunds: add a `transaction_type` column or a separate
   `refund_transactions` table, then convert to absolute values.

4. **Impute or quarantine missing critical fields**  
   For `customer_id` and `amount` (High severity): attempt retrieval from source
   system within an SLA. If unresolved, move records to a staging/quarantine table
   and exclude from reporting until fixed.

5. **Standardise categorical fields**  
   - `currency` → uppercase ISO 4217 (3-letter); validate against a reference lookup  
   - `region` → uppercase ISO 3166-1 alpha-2  
   - `payment_method` → lowercase  
   Apply consistently in the ETL transform layer, not post-hoc.

6. **Flag and review amount outliers**  
   Apply IQR-based detection: flag values beyond `Q3 + 3 × IQR` for business review.
   Route flagged rows to a `review_queue` table; include in reports only after sign-off.

## Reflection Questions

**1. Which KPI gave the strongest signal?**  
The *Validity Error Rate* at ~36.4% is the strongest signal. Over a third of records
violate at least one validity rule, revealing a complete absence of input validation
at the ingestion stage.

**2. Which issue should be escalated first?**  
The duplicate `transaction_id` (T0006) should be escalated first. Duplicate
transactions directly inflate revenue figures — a financial accuracy issue with
immediate business impact — and may indicate a systemic ETL bug affecting records
not yet visible in this sample.

**3. What additional metadata would improve this audit?**  
- A **customer master table** to validate referential integrity (every `customer_id`
  should reference a known customer)  
- A **currency-conversion reference** to cross-check amounts for cross-currency
  anomalies  
- An **ETL job log** (timestamps, source file names, pipeline version) to trace
  when and how each record entered the system, enabling root-cause analysis

## Bonus – Reusable Audit Helper Function

In [ ]:
def run_validation_rules(df, rules):
    """Apply validation rules and return a detailed violation summary.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to validate. Each row is one record.
    rules : dict[str, pd.Series]
        Maps rule name to a boolean mask where True = violation.
        Each mask must share df's index.

    Returns
    -------
    pd.DataFrame
        One row per rule: rule_name, affected_rows, violation_rate,
        example_indices (up to 3 flagged row indices).

    Notes
    -----
    Passing a filtered subset of df (e.g. df[df["region"] == "DE"]) scopes
    the audit to a segment without modifying the function.
    """
    records = []
    for name, mask in rules.items():
        records.append({
            'rule_name':       name,
            'affected_rows':   int(mask.sum()),
            'violation_rate':  f'{mask.mean() * 100:.1f}%',
            'example_indices': df.index[mask].tolist()[:3],
        })
    return (
        pd.DataFrame(records)
        .sort_values('affected_rows', ascending=False)
        .reset_index(drop=True)
    )


# Run on the full dataset
run_validation_rules(df, rules)

### Function Parameter Explanation

- **`df`**: The dataset to validate. Passing the full dataframe audits all records;
  passing a slice (e.g. `df[df['region'] == 'DE']`) scopes the audit to a region
  or cohort without modifying the function itself.
- **`rules`**: A dict mapping descriptive rule names to boolean masks. Adding or
  removing entries changes which checks run, keeping validation logic separate
  from rule definitions. Descriptive names make the output self-documenting.
- Neither parameter has a default because both are required for meaningful output;
  omitting either would produce an empty or misleading result.

## Submission Checklist

- [x] Dataset described with business context and column glossary (Task 1)
- [x] Issues mapped to 5 dimensions: Completeness, Uniqueness, Validity, Consistency, Integrity (Task 2)
- [x] 4 KPIs calculated with targets and interpretation (Task 3)
- [x] 5 validation rules defined with violation counts (Task 4)
- [x] Audit summary with severity and recommended actions (Task 5)
- [x] 6 cleaning recommendations proposed, not implemented (Task 6)
- [x] Bonus reusable helper function with parameter documentation